# Counting Lab: Predict vs Generate (and let the CPU sweat a little)

## 1) What we know (from the literature)
This chapter gives us two complementary superpowers:
- **Counting formulas** (predict how many objects exist without listing them).
- **Generation algorithms** (systematically *produce* permutations/combinations in lexicographic order).

It also suggests computational tasks like generating **r-permutations** and **r-combinations** (with/without repetition). :contentReference[oaicite:2]{index=2}

## 2) The problem we’re curious about
When does “just generate them” become painfully expensive?
And can we **verify** that our generators produce exactly the number of objects the formulas predict?

## 3) The gap (literature → our question)
The text provides pseudocode + formulas, but it doesn’t:
- benchmark runtime growth,
- validate generation counts at multiple sizes,
- show the “blow-up curve” in a way students feel in their bones.

## 4) The work we’ll do
We will:
1. Implement **next-permutation** and **next-r-combination** (lexicographic) based on the chapter algorithms. :contentReference[oaicite:3]{index=3}
2. Build generators that **iterate without storing everything** (so we can go bigger).
3. Compare:
   - predicted counts (math)
   - observed counts (generation)
   - runtime (how much the CPU complains)

## 5) Why we’ll believe the work
We’ll use multiple cross-checks:
- counts match known formulas,
- sequences are strictly increasing in lexicographic order,
- small cases match Python’s trusted `itertools` results.

## Goal: Implement the chapter’s lexicographic "next" building blocks

We’ll implement two primitives:

1) `next_permutation(a)`
   - mutates a permutation list `a` into the next permutation in lexicographic order
   - returns `True` if it advanced, `False` if `a` was already the last permutation

2) `next_r_combination(c, n, r)`
   - mutates a combination list `c` (length r, strictly increasing values in `[0..n-1]`)
   - returns `True` if it advanced, `False` if we hit the last combination

These correspond directly to the chapter’s algorithm blocks for generating the next permutation and next r-combination. :contentReference[oaicite:4]{index=4}

In [2]:
from __future__ import annotations

def next_permutation(a: list[int]) -> bool:
    """
    Transform list a into the next lexicographic permutation.
    Returns True if advanced, False if a was already the last permutation.
    """
    # 1) Find the largest j such that a[j] < a[j+1]
    j = len(a) - 2
    while j >= 0 and a[j] >= a[j + 1]:
        j -= 1
    if j < 0:
        return False  # already the last permutation

    # 2) Find the largest k such that a[j] < a[k]
    k = len(a) - 1
    while a[j] >= a[k]:
        k -= 1

    # 3) Swap a[j], a[k]
    a[j], a[k] = a[k], a[j]

    # 4) Reverse the tail a[j+1:]
    a[j + 1:] = reversed(a[j + 1:])
    return True


def next_r_combination(c: list[int], n: int, r: int) -> bool:
    """
    Transform combination c (0 <= c[0] < ... < c[r-1] < n)
    into the next lexicographic r-combination.
    Returns True if advanced, False if c was already the last combination.
    """
    # Find rightmost position i that can be incremented
    i = r - 1
    while i >= 0 and c[i] == (n - r + i):
        i -= 1
    if i < 0:
        return False  # already the last combination

    # Increment c[i] and reset the tail to the smallest increasing values
    c[i] += 1
    for j in range(i + 1, r):
        c[j] = c[j - 1] + 1
    return True

### Details: how these functions meet the goal

#### `next_permutation(a)`
This is the classic lexicographic stepper:
- Find the "pivot" where the sequence stops increasing from the right.
- Swap pivot with the smallest larger element to its right.
- Reverse the suffix to get the smallest possible tail.

That produces the *next* permutation without skipping or repeating.

#### `next_r_combination(c, n, r)`
An r-combination is a strictly increasing list of indices.
We:
- scan from the right for an index that can still move up,
- increment it,
- then “pack” everything to its right as tightly as possible.

Result: combinations progress in lexicographic order and cover the whole space exactly once.

## Goal: Build generators for permutations and combinations (memory-friendly)

We’ll create:
- `gen_permutations(n)` yielding permutations of `[0..n-1]` in lexicographic order
- `gen_combinations(n, r)` yielding r-combinations of `[0..n-1]`

Key idea:
- We **yield one at a time**, and we never keep the full list in memory.
That lets us push size until the CPU starts writing poetry in heatwaves.

In [3]:
def gen_permutations(n: int):
    a = list(range(n))  # start with smallest lexicographic
    yield tuple(a)
    while next_permutation(a):
        yield tuple(a)

def gen_combinations(n: int, r: int):
    if not (0 <= r <= n):
        return
    c = list(range(r))  # smallest combination
    yield tuple(c)
    while next_r_combination(c, n, r):
        yield tuple(c)

## Goal: Compare predicted counts vs observed counts, and measure runtime

We will compute for a grid of sizes:
- **Predicted count**
  - permutations: `n!`
  - combinations: `C(n, r)`
- **Observed count** by iterating the generator (but not storing outputs)
- **Runtime** with `time.perf_counter()`

Then we’ll plot runtime vs predicted count to visualize blow-up.

This connects the “counting formulas” mindset to the “generation algorithms” mindset,
which the chapter frames as complementary tools for solving counting problems. :contentReference[oaicite:5]{index=5}

In [4]:
import time
import math
import pandas as pd
import matplotlib.pyplot as plt

def consume_count(gen):
    """Count items in a generator without storing them, with a tiny checksum to avoid 'dead code' effects."""
    count = 0
    checksum = 0
    for item in gen:
        count += 1
        # checksum: cheap, deterministic, makes sure we're touching the data
        checksum ^= hash(item)
    return count, checksum

rows = []

# --- combinations benchmark ---
n = 30
for r in [2, 4, 6, 8, 10, 12, 15]:
    predicted = math.comb(n, r)
    t0 = time.perf_counter()
    observed, checksum = consume_count(gen_combinations(n, r))
    t1 = time.perf_counter()
    rows.append({
        "type": "combination",
        "n": n,
        "r": r,
        "predicted_count": predicted,
        "observed_count": observed,
        "seconds": t1 - t0,
        "checksum": checksum
    })

# --- permutations benchmark ---
# Keep this modest by default; n=10 means 3,628,800 permutations (noticeable CPU time).
for n in [7, 8, 9]:
    predicted = math.factorial(n)
    t0 = time.perf_counter()
    observed, checksum = consume_count(gen_permutations(n))
    t1 = time.perf_counter()
    rows.append({
        "type": "permutation",
        "n": n,
        "r": None,
        "predicted_count": predicted,
        "observed_count": observed,
        "seconds": t1 - t0,
        "checksum": checksum
    })

df = pd.DataFrame(rows)
df

,type,n,r,predicted_count,observed_count,seconds,checksum
0,combination,30,2.0,435,435,0.000128,5776175208690956225
1,combination,30,4.0,27405,27405,0.007269,-5238717125775691865
2,combination,30,6.0,593775,593775,0.163844,-5538723321510800773
3,combination,30,8.0,5852925,5852925,1.705700,5443923856122789509
4,combination,30,10.0,30045015,30045015,9.109285,255246889680397120
5,combination,30,12.0,86493225,86493225,28.459209,-3763572419790423436
6,combination,30,15.0,155117520,155117520,61.226432,-8897573278077861521
7,permutation,7,NaN,5040,5040,0.002483,1447764848202905913
8,permutation,8,NaN,40320,40320,0.020459,4344224994681111515
9,permutation,9,NaN,362880,362880,0.226497,-6526065209366183222


In [ ]:
# Plot BOTH types on the same axes
plt.figure()
for t in ["combination", "permutation"]:
    sub = df[df["type"] == t].sort_values("predicted_count")
    plt.plot(sub["predicted_count"], sub["seconds"], marker="o", label=t)

plt.xlabel("Predicted count (number of objects)")
plt.ylabel("Runtime (seconds)")
plt.title("Runtime vs search-space size (generation cost)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Why we believe our work

### 1) Count agreement (the big one)
For each experiment we computed:
- `predicted_count` from math (n!, C(n,r))
- `observed_count` from generation

If the generator is correct, these match exactly.
Any mismatch is a red siren: skipped items, duplicates, or termination errors.

### 2) Structural invariants
- Permutations always contain each number 0..n-1 exactly once.
- Combinations are strictly increasing and stay within 0..n-1.
- Lexicographic progression avoids repeats and guarantees coverage.

### 3) “Independent verification” idea (optional next step)
For small n/r, we can compare our outputs against `itertools.permutations` and `itertools.combinations`.
That gives us a second, trusted implementation to cross-check.

### 4) The lesson students feel
Counting formulas tell us *how many exist*.
Generation algorithms show us *what it costs to enumerate them*.
When the predicted count jumps by 10×, runtime tends to follow, like a loyal (and slightly angry) shadow.

In [ ]:
# HEAT MODE: be kind to your laptop fans 😄
# n=10 => 3,628,800 permutations
# n=11 => 39,916,800 permutations (this can be a LOT in pure Python)

import time, math

n = 10
predicted = math.factorial(n)
print("n =", n, "predicted permutations =", predicted)

t0 = time.perf_counter()
observed, checksum = consume_count(gen_permutations(n))
t1 = time.perf_counter()

print("observed =", observed)
print("seconds  =", t1 - t0)
print("checksum =", checksum)